1. 实现多输入多通道的互相关运算，并验证

In [4]:
import torch
from d2l import torch as d2l

def corr2d_multi_in(X, K):
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))

X = torch.tensor([
    [[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
    [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]
    ])

K = torch.tensor([
    [[0.0, 1.0], [2.0, 3.0]],
    [[1.0, 2.0], [3.0, 4.0]]
    ])

corr2d_multi_in(X, K)

tensor([[ 56.,  72.],
        [104., 120.]])

2. 计算多个通道的输出的 互相关函数

In [5]:
def corr2d_multi_in_out(X, K):
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

K = torch.stack((K, K + 1, K + 2), 0)
K.shape
# 输出通道3，输入通道2，高宽2

torch.Size([3, 2, 2, 2])

In [6]:
corr2d_multi_in_out(X, K)  
# 3x 2x2

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

3. 1x1卷积

In [8]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h * w))
    K = K.reshape((c_o, c_i))
    
    # 保证第一个矩阵的列 = 第二个的行
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))

X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))

Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6